In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
from datasets import load_dataset

dataset = load_dataset("freococo/burmese-contextual-pragmatics")
print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

burmese-conversational-pragmatics.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2200 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['uid', 'root_meaning', 'instruction', 'context', 'style', 'tone', 'utterance', 'phonetic', 'register', 'politeness', 'emotion', 'power_relation', 'intent_strength', 'notes'],
        num_rows: 2200
    })
})


In [ ]:
example = dataset["train"][0]
for key, value in example.items():
    print(f"{key:20s} : {value}")

uid                  : bur_beau_001
root_meaning         : You are so beautiful
instruction          : Simple standard compliment.
context              : General social interaction.
style                : Standard
tone                 : Neutral
utterance            : မင်း တကယ် လှတယ်။
phonetic             : Min ta-kal hla tal
register             : colloquial
politeness           : neutral
emotion              : admiration
power_relation       : equal
intent_strength      : normal
notes                : The most common way to say 'You are really beautiful'.


In [ ]:
from collections import Counter

label_col = "politeness"

counts = Counter(dataset["train"][label_col])
total = len(dataset["train"])

print(f"Label: '{label_col}'")
print(f"Total examples: {total}\n")
for label, count in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 20)
    print(f"{label:20s} {count:4d}  {bar}")

Label: 'politeness'
Total examples: 2200

neutral              1165  ██████████████████████████████████████████████████████████
polite                373  ██████████████████
informal              341  █████████████████
professional           94  ████
blunt                  70  ███
rude                   53  ██
very_polite            49  ██
friendly               33  █
religious              11  
impolite                7  
stern                   2  
sarcastic               1  
polite_but_firm         1  


In [ ]:
print(set(dataset["train"]["politeness"]))

{'blunt', 'professional', 'impolite', 'rude', 'friendly', 'very_polite', 'stern', 'polite', 'informal', 'sarcastic', 'religious', 'neutral', 'polite_but_firm'}


In [ ]:
# Labels with very few examples will hurt training and evaluation.
# We merge them into the closest sensible category.

label_mapping = {
    "neutral":        "neutral",
    "polite":         "polite",
    "informal":       "informal",
    "professional":   "professional",
    "blunt":          "blunt",
    "rude":           "rude",
    "very_polite":    "polite",        # merge into polite
    "friendly":       "informal",      # merge into informal
    "religious":      "polite",        # formal/respectful → polite
    "impolite":       "rude",          # merge into rude
    "stern":          "blunt",         # merge into blunt
    "sarcastic":      "rude",          # merge into rude
    "polite_but_firm":"polite",        # merge into polite
}

def apply_label_mapping(example):
    example["politeness_coarse"] = label_mapping[example["politeness"]]
    return example

dataset = dataset.map(apply_label_mapping)

# Verify the result
from collections import Counter
counts = Counter(dataset["train"]["politeness_coarse"])
print("Coarsened label distribution:\n")
for label, count in sorted(counts.items(), key=lambda x: -x[1]):
    bar = "█" * (count // 20)
    print(f"{label:20s} {count:4d}  {bar}")

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Coarsened label distribution:

neutral              1165  ██████████████████████████████████████████████████████████
polite                434  █████████████████████
informal              374  ██████████████████
professional           94  ████
blunt                  72  ███
rude                   61  ███


In [ ]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

le = LabelEncoder()
all_labels = dataset["train"]["politeness_coarse"]
le.fit(all_labels)

print("Label classes:", le.classes_)
print("Example encoding:")
for label in le.classes_:
    print(f"  {label:20s} → {le.transform([label])[0]}")

Label classes: ['blunt' 'informal' 'neutral' 'polite' 'professional' 'rude']
Example encoding:
  blunt                → 0
  informal             → 1
  neutral              → 2
  polite               → 3
  professional         → 4
  rude                 → 5


In [ ]:
def encode_labels(example):
    example["label"] = int(le.transform([example["politeness_coarse"]])[0])
    return example

dataset = dataset.map(encode_labels)

# Split: 70% train, 15% validation, 15% test
split1 = dataset["train"].train_test_split(test_size=0.30, seed=42)
split2 = split1["test"].train_test_split(test_size=0.50, seed=42)

train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train:      {len(train_data)} examples")
print(f"Validation: {len(val_data)} examples")
print(f"Test:       {len(test_data)} examples")

Map:   0%|          | 0/2200 [00:00<?, ? examples/s]

Train:      1540 examples
Validation: 330 examples
Test:       330 examples


In [ ]:
from collections import Counter

for name, split in [("Train", train_data), ("Val", val_data), ("Test", test_data)]:
    counts = Counter(split["politeness_coarse"])
    print(f"\n{name} ({len(split)} rows):")
    for label, count in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {label:20s} {count}")


Train (1540 rows):
  neutral              806
  polite               314
  informal             263
  professional         66
  blunt                49
  rude                 42

Val (330 rows):
  neutral              185
  polite               64
  informal             52
  professional         13
  blunt                9
  rude                 7

Test (330 rows):
  neutral              174
  informal             59
  polite               56
  professional         15
  blunt                14
  rude                 12


In [ ]:
from transformers import AutoTokenizer

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded:", type(tokenizer).__name__)
print("Vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded: XLMRobertaTokenizer
Vocab size: 250002


In [ ]:
sample_text = dataset["train"][0]["utterance"]
print("Original text:", sample_text)

tokens = tokenizer(sample_text, return_tensors=None)
print("Token IDs:", tokens["input_ids"])
print("Decoded back:", tokenizer.decode(tokens["input_ids"]))
print("Number of tokens:", len(tokens["input_ids"]))

Original text: မင်း တကယ် လှတယ်။
Token IDs: [0, 6, 65644, 14725, 92186, 6, 35806, 55778, 2]
Decoded back: <s> မင်း တကယ် လှတယ်။</s>
Number of tokens: 9


In [ ]:
MAX_LENGTH = 128

def tokenize_utterance_only(example):
    encoding = tokenizer(
        example["utterance"],
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    encoding["label"] = example["label"]
    return encoding

In [ ]:
cols_to_remove = [c for c in train_data.column_names if c not in ["input_ids", "attention_mask", "label"]]

tokenized_train = train_data.map(tokenize_utterance_only, remove_columns=cols_to_remove)
tokenized_val   = val_data.map(tokenize_utterance_only,   remove_columns=cols_to_remove)
tokenized_test  = test_data.map(tokenize_utterance_only,  remove_columns=cols_to_remove)

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")
tokenized_test.set_format("torch")

print("Columns kept:", tokenized_train.column_names)
print("Sample input_ids shape:", tokenized_train[0]["input_ids"].shape)
print("Sample label:", tokenized_train[0]["label"])

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Columns kept: ['label', 'input_ids', 'attention_mask']
Sample input_ids shape: torch.Size([128])
Sample label: tensor(1)


In [ ]:
import torch
from collections import Counter

label_counts = Counter(train_data["label"])
total = len(train_data)
num_classes = len(le.classes_)

# Weight for each class = total / (num_classes * count)
# This makes rare classes matter more during training
weights = [total / (num_classes * label_counts[i]) for i in range(num_classes)]
class_weights = torch.tensor(weights, dtype=torch.float)

print("Class weights:")
for i, (label, weight) in enumerate(zip(le.classes_, weights)):
    print(f"  {label:20s} (class {i})  count: {label_counts[i]:4d}  weight: {weight:.4f}")

Class weights:
  blunt                (class 0)  count:   49  weight: 5.2381
  informal             (class 1)  count:  263  weight: 0.9759
  neutral              (class 2)  count:  806  weight: 0.3184
  polite               (class 3)  count:  314  weight: 0.8174
  professional         (class 4)  count:   66  weight: 3.8889
  rude                 (class 5)  count:   42  weight: 6.1111


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

print("Model loaded:", type(model).__name__)
print("Number of labels:", num_classes)
print("Label order:", list(le.classes_))

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: XLMRobertaForSequenceClassification
Number of labels: 6
Label order: [np.str_('blunt'), np.str_('informal'), np.str_('neutral'), np.str_('polite'), np.str_('professional'), np.str_('rude')]


In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Move weights to same device as logits
        weights = class_weights.to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    weighted_f1 = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": round(accuracy, 4),
        "macro_f1": round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4)
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./xlmr_stage1_utterance_only",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.802056,1.835582,0.221200,0.101100,0.132400
2,1.802321,1.646328,0.315200,0.207200,0.282600
3,1.632802,1.420563,0.451500,0.327700,0.455300
4,1.339599,1.201276,0.518200,0.475400,0.557700
5,1.249163,1.140586,0.542400,0.517100,0.548100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=485, training_loss=1.5554918584135389, metrics={'train_runtime': 448.7274, 'train_samples_per_second': 17.16, 'train_steps_per_second': 1.081, 'total_flos': 506506971801600.0, 'train_loss': 1.5554918584135389, 'epoch': 5.0})

In [ ]:
# Evaluate on test set with current best model
results = trainer.evaluate(tokenized_test)
print("\nStage 1 Test Results (utterance only):")
for k, v in results.items():
    print(f"  {k}: {v}")


Stage 1 Test Results (utterance only):
  eval_loss: 1.1449801921844482
  eval_accuracy: 0.5515
  eval_macro_f1: 0.484
  eval_weighted_f1: 0.5695
  eval_runtime: 2.3544
  eval_samples_per_second: 140.165
  eval_steps_per_second: 4.672
  epoch: 5.0


In [ ]:
# Continue training from best checkpoint
training_args_continued = TrainingArguments(
    output_dir="./xlmr_stage1_utterance_only",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,         # lower LR since we're continuing
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42
)

trainer_continued = WeightedTrainer(
    model=trainer.model,        # reuse the already-trained model
    args=training_args_continued,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

trainer_continued.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.154066,1.109480,0.524200,0.476600,0.528800
2,1.050772,1.006341,0.545500,0.490900,0.560000
3,0.994081,1.013678,0.545500,0.474000,0.559400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=291, training_loss=1.0836633374191231, metrics={'train_runtime': 238.1451, 'train_samples_per_second': 19.4, 'train_steps_per_second': 1.222, 'total_flos': 303904183080960.0, 'train_loss': 1.0836633374191231, 'epoch': 3.0})

In [ ]:
results_s1_final = trainer_continued.evaluate(tokenized_test)
print("\nStage 1 FINAL Test Results (utterance only):")
for k, v in results_s1_final.items():
    print(f"  {k}: {v}")


Stage 1 FINAL Test Results (utterance only):
  eval_loss: 1.0432374477386475
  eval_accuracy: 0.5758
  eval_macro_f1: 0.5482
  eval_weighted_f1: 0.5886
  eval_runtime: 2.1251
  eval_samples_per_second: 155.284
  eval_steps_per_second: 5.176
  epoch: 3.0


In [ ]:
def tokenize_with_context(example):
    # Combine utterance + context + instruction into one input string
    # </s> is XLM-R's separator token — tells the model these are distinct pieces
    text = (
        example["utterance"]
        + " </s> "
        + str(example["context"]).strip()
        + " </s> "
        + str(example["instruction"]).strip()
    )

    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    encoding["label"] = example["label"]
    return encoding

In [ ]:
tokenized_train_s2 = train_data.map(tokenize_with_context, remove_columns=cols_to_remove)
tokenized_val_s2   = val_data.map(tokenize_with_context,   remove_columns=cols_to_remove)
tokenized_test_s2  = test_data.map(tokenize_with_context,  remove_columns=cols_to_remove)

tokenized_train_s2.set_format("torch")
tokenized_val_s2.set_format("torch")
tokenized_test_s2.set_format("torch")

# Verify the new input looks right
sample_idx = 0
ids = tokenized_train_s2[sample_idx]["input_ids"]
print("Decoded Stage 2 input sample:")
print(tokenizer.decode(ids, skip_special_tokens=False))

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Decoded Stage 2 input sample:
<s> ဗိုက်ထဲက တဂွီဂွီ မြည်နေပြီ။</s> To a friend.</s> Express hunger through stomach sounds.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


In [ ]:
from transformers import AutoModelForSequenceClassification

model_s2 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

print("Fresh Stage 2 model loaded.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh Stage 2 model loaded.


In [ ]:
training_args_s2 = TrainingArguments(
    output_dir="./xlmr_stage2_with_context",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=150,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42
)

trainer_s2 = WeightedTrainer(
    model=model_s2,
    args=training_args_s2,
    train_dataset=tokenized_train_s2,
    eval_dataset=tokenized_val_s2,
    compute_metrics=compute_metrics
)

trainer_s2.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.805088,1.754127,0.160600,0.065100,0.065800
2,1.749269,1.442746,0.645500,0.391200,0.619300
3,1.313582,1.091696,0.600000,0.476100,0.608400
4,0.870179,0.915675,0.645500,0.561300,0.671100
5,0.769031,1.023414,0.693900,0.620500,0.701200
6,0.558001,0.997383,0.672700,0.543200,0.685100
7,0.473844,1.035995,0.721200,0.574000,0.729100
8,0.388958,0.987046,0.715200,0.617400,0.721300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=776, training_loss=0.9834659025841153, metrics={'train_runtime': 675.9736, 'train_samples_per_second': 18.226, 'train_steps_per_second': 1.148, 'total_flos': 810411154882560.0, 'train_loss': 0.9834659025841153, 'epoch': 8.0})

In [ ]:
results_s2 = trainer_s2.evaluate(tokenized_test_s2)
print("\nStage 2 Test Results (utterance + context + instruction):")
for k, v in results_s2.items():
    print(f"  {k}: {v}")


Stage 2 Test Results (utterance + context + instruction):
  eval_loss: 0.7066054940223694
  eval_accuracy: 0.7485
  eval_macro_f1: 0.7056
  eval_weighted_f1: 0.7529
  eval_runtime: 2.2004
  eval_samples_per_second: 149.973
  eval_steps_per_second: 4.999
  epoch: 8.0


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import torch

# Get predictions on test set
predictions_output = trainer_s2.predict(tokenized_test_s2)
preds = np.argmax(predictions_output.predictions, axis=1)
true_labels = predictions_output.label_ids

# Full classification report
print("Stage 2 Classification Report:")
print(classification_report(
    true_labels,
    preds,
    target_names=le.classes_
))

Stage 2 Classification Report:
              precision    recall  f1-score   support

       blunt       0.60      0.64      0.62        14
    informal       0.69      0.76      0.73        59
     neutral       0.89      0.73      0.80       174
      polite       0.58      0.80      0.68        56
professional       0.83      1.00      0.91        15
        rude       0.50      0.50      0.50        12

    accuracy                           0.75       330
   macro avg       0.68      0.74      0.71       330
weighted avg       0.77      0.75      0.75       330



In [ ]:
def tokenize_full(example):
    # Add register and power_relation as explicit text prefix
    # These are structured pragmatic signals the model didn't see in Stage 2
    prefix = (
        f"[register: {example['register']}] "
        f"[power: {example['power_relation']}] "
        f"[tone: {example['tone']}] "
    )

    text = (
        prefix
        + example["utterance"]
        + " </s> "
        + str(example["context"]).strip()
        + " </s> "
        + str(example["instruction"]).strip()
    )

    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    encoding["label"] = example["label"]
    return encoding

In [ ]:
tokenized_train_s3 = train_data.map(tokenize_full, remove_columns=cols_to_remove)
tokenized_val_s3   = val_data.map(tokenize_full,   remove_columns=cols_to_remove)
tokenized_test_s3  = test_data.map(tokenize_full,  remove_columns=cols_to_remove)

tokenized_train_s3.set_format("torch")
tokenized_val_s3.set_format("torch")
tokenized_test_s3.set_format("torch")

# Check what the full input looks like
sample_ids = tokenized_train_s3[0]["input_ids"]
print("Stage 3 input sample:")
print(tokenizer.decode(sample_ids, skip_special_tokens=False))

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Stage 3 input sample:
<s> [register: colloquial] [power: equal] [tone: Neutral] ဗိုက်ထဲက တဂွီဂွီ မြည်နေပြီ။</s> To a friend.</s> Express hunger through stomach sounds.</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>


In [ ]:
import torch
import gc

# Delete previous models from GPU
del model
del model_s2
gc.collect()
torch.cuda.empty_cache()

# Check how much memory is free now
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 1e9:.2f} GB / {total / 1e9:.2f} GB")

GPU memory free: 2.15 GB / 15.64 GB


In [ ]:
import torch, gc

# Kill everything that might be holding GPU memory
for var in ['model', 'model_s2', 'model_s3', 'trainer', 'trainer_s2',
            'trainer_s3', 'trainer_continued']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
torch.cuda.empty_cache()

# Also clear the predictions objects
for var in ['predictions_output', 'predictions_s3']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
torch.cuda.empty_cache()

free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 1e9:.2f} GB / {total / 1e9:.2f} GB")

GPU memory free: 11.67 GB / 15.64 GB


In [ ]:
model_s3 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

training_args_s3 = TrainingArguments(
    output_dir="./xlmr_stage3_full",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=150,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42
)

trainer_s3 = WeightedTrainer(
    model=model_s3,
    args=training_args_s3,
    train_dataset=tokenized_train_s3,
    eval_dataset=tokenized_val_s3,
    compute_metrics=compute_metrics
)

trainer_s3.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.785111,1.684267,0.472700,0.259200,0.480100
2,1.578838,0.874048,0.690900,0.531900,0.704500
3,0.986446,0.783404,0.730300,0.658100,0.734500
4,0.697513,0.582801,0.809100,0.736800,0.814500
5,0.544027,0.525146,0.842400,0.783300,0.845800
6,0.415120,0.663788,0.848500,0.724200,0.845300
7,0.316791,0.489080,0.860600,0.805900,0.862500
8,0.284293,0.504565,0.860600,0.819900,0.862700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=776, training_loss=0.7837099912240333, metrics={'train_runtime': 814.6382, 'train_samples_per_second': 15.123, 'train_steps_per_second': 0.953, 'total_flos': 810411154882560.0, 'train_loss': 0.7837099912240333, 'epoch': 8.0})

In [ ]:
results_s3 = trainer_s3.evaluate(tokenized_test_s3)
print("\nStage 3 Test Results (full pragmatic input):")
for k, v in results_s3.items():
    print(f"  {k}: {v}")



Stage 3 Test Results (full pragmatic input):
  eval_loss: 0.5389723181724548
  eval_accuracy: 0.8394
  eval_macro_f1: 0.8252
  eval_weighted_f1: 0.8421
  eval_runtime: 2.211
  eval_samples_per_second: 149.252
  eval_steps_per_second: 4.975
  epoch: 8.0


In [ ]:
predictions_s3 = trainer_s3.predict(tokenized_test_s3)
preds_s3 = np.argmax(predictions_s3.predictions, axis=1)
true_s3 = predictions_s3.label_ids

print("Stage 3 Classification Report:")
print(classification_report(true_s3, preds_s3, target_names=le.classes_))

Stage 3 Classification Report:
              precision    recall  f1-score   support

       blunt       0.83      0.71      0.77        14
    informal       0.84      0.90      0.87        59
     neutral       0.93      0.82      0.87       174
      polite       0.66      0.86      0.74        56
professional       0.79      1.00      0.88        15
        rude       0.90      0.75      0.82        12

    accuracy                           0.84       330
   macro avg       0.82      0.84      0.83       330
weighted avg       0.86      0.84      0.84       330



In [ ]:
from google.colab import files
import zipfile, os, pickle, json
import numpy as np
from sklearn.metrics import classification_report

# ── 1. Save all model checkpoints ──────────────────────────────────────────

def zip_and_download(folder, zip_name):
    if os.path.exists(folder):
        with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, filenames in os.walk(folder):
                for filename in filenames:
                    filepath = os.path.join(root, filename)
                    zipf.write(filepath)
        print(f"Downloading {zip_name}...")
        files.download(zip_name)
    else:
        print(f"Skipping {zip_name} — folder not found")

zip_and_download("./xlmr_stage1_utterance_only",  "xlmr_stage1.zip")
zip_and_download("./xlmr_stage2_with_context",     "xlmr_stage2.zip")
zip_and_download("./xlmr_stage3_full",             "xlmr_stage3.zip")

# ── 2. Save label encoder ───────────────────────────────────────────────────

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("Saved label_encoder.pkl")
files.download("label_encoder.pkl")

# ── 3. Save label mapping (coarsening map) ──────────────────────────────────

with open("label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)
print("Saved label_mapping.json")
files.download("label_mapping.json")

# ── 4. Save all numerical results ───────────────────────────────────────────

all_results = {
    "stage1": {
        "description": "Utterance only — no context",
        "test_accuracy":    0.5758,
        "test_macro_f1":    0.5482,
        "test_weighted_f1": 0.5886,
        "test_loss":        1.0432
    },
    "stage2": {
        "description": "Utterance + context + instruction",
        "test_accuracy":    0.7485,
        "test_macro_f1":    0.7056,
        "test_weighted_f1": 0.7529,
        "test_loss":        0.7066
    },
    "stage3": {
        "description": "Register + power + tone prefix + utterance + context + instruction",
        "test_accuracy":    0.8394,
        "test_macro_f1":    0.8252,
        "test_weighted_f1": 0.8421,
        "test_loss":        0.5390
    }
}

with open("all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
print("Saved all_results.json")
files.download("all_results.json")

# ── 5. Save classification reports ──────────────────────────────────────────

# Stage 2
report_s2 = classification_report(
    true_labels, preds,
    target_names=le.classes_,
    output_dict=True
)
with open("classification_report_stage2.json", "w") as f:
    json.dump(report_s2, f, indent=2)
print("Saved classification_report_stage2.json")
files.download("classification_report_stage2.json")

# Stage 3
report_s3 = classification_report(
    true_s3, preds_s3,
    target_names=le.classes_,
    output_dict=True
)
with open("classification_report_stage3.json", "w") as f:
    json.dump(report_s3, f, indent=2)
print("Saved classification_report_stage3.json")
files.download("classification_report_stage3.json")

# ── 6. Save dataset splits as CSV ───────────────────────────────────────────

train_data.to_csv("split_train.csv", index=False)
val_data.to_csv("split_val.csv",     index=False)
test_data.to_csv("split_test.csv",   index=False)
print("Saved dataset splits")
files.download("split_train.csv")
files.download("split_val.csv")
files.download("split_test.csv")

# ── 7. Save training curves ──────────────────────────────────────────────────

training_curves = {
    "stage1": {
        "epochs": [1, 2, 3, 4, 5, 6, 7, 8],
        "train_loss":  [1.802056, 1.802321, 1.632802, 1.339599, 1.249163, 1.154066, 1.050772, 0.994081],
        "val_loss":    [1.835582, 1.646328, 1.420563, 1.201276, 1.140586, 1.109480, 1.006341, 1.013678],
        "macro_f1":    [0.1011,   0.2072,   0.3277,   0.4754,   0.5171,   0.4766,   0.4909,   0.4740]
    },
    "stage2": {
        "epochs": [1, 2, 3, 4, 5, 6, 7, 8],
        "train_loss":  [1.805088, 1.749269, 1.313582, 0.870179, 0.769031, 0.558001, 0.473844, 0.388958],
        "val_loss":    [1.754127, 1.442746, 1.091696, 0.915675, 1.023414, 0.997383, 1.035995, 0.987046],
        "macro_f1":    [0.0651,   0.3912,   0.4761,   0.5613,   0.6205,   0.5432,   0.5740,   0.6174]
    },
    "stage3": {
        "epochs": [1, 2, 3, 4, 5, 6, 7, 8],
        "train_loss":  [1.785111, 1.578838, 0.986446, 0.697513, 0.544027, 0.415120, 0.316791, 0.284293],
        "val_loss":    [1.684267, 0.874048, 0.783404, 0.582801, 0.525146, 0.663788, 0.489080, 0.504565],
        "macro_f1":    [0.2592,   0.5319,   0.6581,   0.7368,   0.7833,   0.7242,   0.8059,   0.8199]
    }
}

with open("training_curves.json", "w") as f:
    json.dump(training_curves, f, indent=2)
print("Saved training_curves.json")
files.download("training_curves.json")

# ── Done ─────────────────────────────────────────────────────────────────────
print("\nAll files saved and downloading. Check your browser download queue.")
print("Files to expect:")
print("  xlmr_stage1.zip, xlmr_stage2.zip, xlmr_stage3.zip")
print("  label_encoder.pkl, label_mapping.json")
print("  all_results.json, training_curves.json")
print("  classification_report_stage2.json, classification_report_stage3.json")
print("  split_train.csv, split_val.csv, split_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

OSError: [Errno 28] No space left on device

In [ ]:
import os

for zip_name in ["xlmr_stage1.zip", "xlmr_stage2.zip"]:
    if os.path.exists(zip_name):
        size_mb = os.path.getsize(zip_name) / 1e6
        print(f"{zip_name}: {size_mb:.1f} MB")
    else:
        print(f"{zip_name}: NOT FOUND")

# Also check what's eating the disk
import shutil
total, used, free = shutil.disk_usage("/")
print(f"\nDisk: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total / {free/1e9:.1f} GB free")

xlmr_stage1.zip: 7256.6 MB
xlmr_stage2.zip: 7348.0 MB

Disk: 120.9 GB used / 120.9 GB total / 0.0 GB free


In [ ]:
from google.colab import files
import pickle, json

# ── Label encoder ───────────────────────────────────────────────────────────
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
files.download("label_encoder.pkl")

# ── Label mapping ────────────────────────────────────────────────────────────
with open("label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)
files.download("label_mapping.json")

# ── All numerical results ────────────────────────────────────────────────────
all_results = {
    "stage1": {"description": "Utterance only", "accuracy": 0.5758, "macro_f1": 0.5482, "weighted_f1": 0.5886, "loss": 1.0432},
    "stage2": {"description": "Utterance + context + instruction", "accuracy": 0.7485, "macro_f1": 0.7056, "weighted_f1": 0.7529, "loss": 0.7066},
    "stage3": {"description": "Register + power + tone + utterance + context + instruction", "accuracy": 0.8394, "macro_f1": 0.8252, "weighted_f1": 0.8421, "loss": 0.5390}
}
with open("all_results.json", "w") as f:
    json.dump(all_results, f, indent=2)
files.download("all_results.json")

# ── Training curves ──────────────────────────────────────────────────────────
training_curves = {
    "stage1": {
        "epochs":      [1,2,3,4,5,6,7,8],
        "train_loss":  [1.802,1.802,1.633,1.340,1.249,1.154,1.051,0.994],
        "val_loss":    [1.836,1.646,1.421,1.201,1.141,1.109,1.006,1.014],
        "macro_f1":    [0.101,0.207,0.328,0.475,0.517,0.477,0.491,0.474]
    },
    "stage2": {
        "epochs":      [1,2,3,4,5,6,7,8],
        "train_loss":  [1.805,1.749,1.314,0.870,0.769,0.558,0.474,0.389],
        "val_loss":    [1.754,1.443,1.092,0.916,1.023,0.997,1.036,0.987],
        "macro_f1":    [0.065,0.391,0.476,0.561,0.621,0.543,0.574,0.617]
    },
    "stage3": {
        "epochs":      [1,2,3,4,5,6,7,8],
        "train_loss":  [1.785,1.579,0.986,0.698,0.544,0.415,0.317,0.284],
        "val_loss":    [1.684,0.874,0.783,0.583,0.525,0.664,0.489,0.505],
        "macro_f1":    [0.259,0.532,0.658,0.737,0.783,0.724,0.806,0.820]
    }
}
with open("training_curves.json", "w") as f:
    json.dump(training_curves, f, indent=2)
files.download("training_curves.json")

# ── Classification reports ───────────────────────────────────────────────────
from sklearn.metrics import classification_report
import numpy as np

report_s2 = classification_report(true_labels, preds, target_names=le.classes_, output_dict=True)
with open("report_stage2.json", "w") as f:
    json.dump(report_s2, f, indent=2)
files.download("report_stage2.json")

report_s3 = classification_report(true_s3, preds_s3, target_names=le.classes_, output_dict=True)
with open("report_stage3.json", "w") as f:
    json.dump(report_s3, f, indent=2)
files.download("report_stage3.json")

# ── Dataset splits ───────────────────────────────────────────────────────────
train_data.to_csv("split_train.csv", index=False)
val_data.to_csv("split_val.csv",     index=False)
test_data.to_csv("split_test.csv",   index=False)
files.download("split_train.csv")
files.download("split_val.csv")
files.download("split_test.csv")

print("\nAll lightweight files queued for download:")
print("  label_encoder.pkl, label_mapping.json")
print("  all_results.json, training_curves.json")
print("  report_stage2.json, report_stage3.json")
print("  split_train.csv, split_val.csv, split_test.csv")

OSError: [Errno 28] No space left on device

In [ ]:
import os, shutil, gc

# Force delete the zip files if still lingering
for f in ["xlmr_stage1.zip", "xlmr_stage2.zip", "xlmr_stage3.zip"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")

# Also delete any other temp files that might be around
for f in ["label_encoder.pkl", "label_mapping.json", "all_results.json",
          "training_curves.json", "report_stage2.json", "report_stage3.json",
          "split_train.csv", "split_val.csv", "split_test.csv"]:
    if os.path.exists(f):
        os.remove(f)

gc.collect()

# Check disk now
total, used, free = shutil.disk_usage("/")
print(f"\nDisk: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total")
print(f"Free: {free/1e9:.1f} GB")


Disk: 120.9 GB used / 120.9 GB total
Free: 0.0 GB


In [ ]:
from huggingface_hub import login
login()

In [ ]:
import shutil, gc

shutil.rmtree("./xlmr_stage1_utterance_only", ignore_errors=True)
shutil.rmtree("./xlmr_stage2_with_context", ignore_errors=True)
shutil.rmtree("./xlmr_stage3_full", ignore_errors=True)
gc.collect()

total, used, free = shutil.disk_usage("/")
print(f"Free: {free/1e9:.1f} GB")

Free: 56.7 GB


In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments
from transformers import Trainer
import torch.nn as nn

# Fresh model
model_s1 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

training_args_s1 = TrainingArguments(
    output_dir="annasus10/xlmr-burmese-pragmatics-stage1",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=150,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42,
    push_to_hub=True,
    hub_model_id="annasus10/xlmr-burmese-pragmatics-stage1",
    hub_strategy="every_save"
)

trainer_s1 = WeightedTrainer(
    model=model_s1,
    args=training_args_s1,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)

trainer_s1.train()
trainer_s1.push_to_hub()
print("Stage 1 pushed to HuggingFace.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.808079,1.785928,0.157600,0.045400,0.042900
2,1.804730,1.809429,0.160600,0.048600,0.054400
3,1.801494,1.734680,0.290900,0.131100,0.236100
4,1.702344,1.621747,0.593900,0.262900,0.463400
5,1.474196,1.275796,0.521200,0.474200,0.526300
6,1.252190,1.162686,0.560600,0.494700,0.568100
7,1.166179,1.108756,0.548500,0.545100,0.548800
8,1.027138,1.091269,0.572700,0.534200,0.582800


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-stage1/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...-stage1/model.safetensors:  11%|#1        |  128MB / 1.11GB            

Stage 1 pushed to HuggingFace.


In [ ]:
# Login first
from huggingface_hub import login
login()

In [ ]:
def tokenize_with_context(example):
    text = (
        example["utterance"]
        + " </s> "
        + str(example["context"]).strip()
        + " </s> "
        + str(example["instruction"]).strip()
    )
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    encoding["label"] = example["label"]
    return encoding

cols_to_remove = [c for c in train_data.column_names if c not in ["input_ids", "attention_mask", "label"]]

tokenized_train_s2 = train_data.map(tokenize_with_context, remove_columns=cols_to_remove)
tokenized_val_s2   = val_data.map(tokenize_with_context,   remove_columns=cols_to_remove)
tokenized_test_s2  = test_data.map(tokenize_with_context,  remove_columns=cols_to_remove)

tokenized_train_s2.set_format("torch")
tokenized_val_s2.set_format("torch")
tokenized_test_s2.set_format("torch")

print("Stage 2 tokenization done.")

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Stage 2 tokenization done.


In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments
from transformers import Trainer
import torch.nn as nn

model_s2 = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=num_classes
)

training_args_s2 = TrainingArguments(
    output_dir="annasus10/xlmr-burmese-pragmatics-stage2",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=150,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42,
    push_to_hub=True,
    hub_model_id="annasus10/xlmr-burmese-pragmatics-stage2",
    hub_strategy="every_save"
)

trainer_s2 = WeightedTrainer(
    model=model_s2,
    args=training_args_s2,
    train_dataset=tokenized_train_s2,
    eval_dataset=tokenized_val_s2,
    compute_metrics=compute_metrics
)

trainer_s2.train()
trainer_s2.push_to_hub()
print("Stage 2 pushed to HuggingFace.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,1.794855,1.775633,0.160600,0.050800,0.049100
2,1.700861,1.408634,0.493900,0.375400,0.514400
3,1.262901,1.065500,0.581800,0.491000,0.581100
4,0.830216,0.842846,0.681800,0.579300,0.692300
5,0.645812,0.853406,0.751500,0.651300,0.754000
6,0.539041,0.948201,0.748500,0.619800,0.751500
7,0.407361,0.903552,0.733300,0.636200,0.737000
8,0.331261,0.871295,0.760600,0.649800,0.764000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-stage2/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...-stage2/model.safetensors:  14%|#3        |  152MB / 1.11GB            

Stage 2 pushed to HuggingFace.


In [ ]:
# ── Stage 3 Tokenization ─────────────────────────────────────────────────────

def tokenize_full(example):
    prefix = (
        f"[register: {example['register']}] "
        f"[power: {example['power_relation']}] "
        f"[tone: {example['tone']}] "
    )
    text = (
        prefix
        + example["utterance"]
        + " </s> "
        + str(example["context"]).strip()
        + " </s> "
        + str(example["instruction"]).strip()
    )
    encoding = tokenizer(
        text,
        max_length=MAX_LENGTH,
        truncation=True,
        padding="max_length"
    )
    encoding["label"] = example["label"]
    return encoding

tokenized_train_s3 = train_data.map(tokenize_full, remove_columns=cols_to_remove)
tokenized_val_s3   = val_data.map(tokenize_full,   remove_columns=cols_to_remove)
tokenized_test_s3  = test_data.map(tokenize_full,  remove_columns=cols_to_remove)

tokenized_train_s3.set_format("torch")
tokenized_val_s3.set_format("torch")
tokenized_test_s3.set_format("torch")

print("Stage 3 tokenization done.")

# ── Stage 3 Training ──────────────────────────────────────────────────────────

import gc
import torch

# Clear Stage 2 model from GPU before loading Stage 3
del model_s2
gc.collect()
torch.cuda.empty_cache()

model_s3 = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

training_args_s3 = TrainingArguments(
    output_dir="annasus10/xlmr-burmese-pragmatics-stage3",
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=150,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
    seed=42,
    push_to_hub=True,
    hub_model_id="annasus10/xlmr-burmese-pragmatics-stage3",
    hub_strategy="every_save"
)

trainer_s3 = WeightedTrainer(
    model=model_s3,
    args=training_args_s3,
    train_dataset=tokenized_train_s3,
    eval_dataset=tokenized_val_s3,
    compute_metrics=compute_metrics
)

trainer_s3.train()
trainer_s3.push_to_hub()
print("Stage 3 pushed to HuggingFace.")

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Map:   0%|          | 0/330 [00:00<?, ? examples/s]

Stage 3 tokenization done.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-stage3/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...-stage3/model.safetensors:  12%|#2        |  135MB / 1.11GB            

Stage 3 pushed to HuggingFace.


In [25]:
# Stage 2 test evaluation
results_s2 = trainer_s2.evaluate(tokenized_test_s2)
print("\nStage 2 Test Results:")
for k, v in results_s2.items():
    print(f"  {k}: {v}")


Stage 2 Test Results:
  eval_loss: 0.8274039626121521
  eval_accuracy: 0.7515
  eval_macro_f1: 0.728
  eval_weighted_f1: 0.7544
  eval_runtime: 2.2299
  eval_samples_per_second: 147.988
  eval_steps_per_second: 4.933
  epoch: 8.0


In [26]:
# Stage 2 classification report
import numpy as np
from sklearn.metrics import classification_report

predictions_s2 = trainer_s2.predict(tokenized_test_s2)
preds_s2 = np.argmax(predictions_s2.predictions, axis=1)
true_s2 = predictions_s2.label_ids

print("Stage 2 Classification Report:")
print(classification_report(true_s2, preds_s2, target_names=le.classes_))

Stage 2 Classification Report:
              precision    recall  f1-score   support

       blunt       0.80      0.57      0.67        14
    informal       0.71      0.76      0.74        59
     neutral       0.86      0.73      0.79       174
      polite       0.59      0.82      0.69        56
professional       0.78      0.93      0.85        15
        rude       0.62      0.67      0.64        12

    accuracy                           0.75       330
   macro avg       0.73      0.75      0.73       330
weighted avg       0.77      0.75      0.75       330



In [27]:
from transformers import AutoModelForSequenceClassification
import numpy as np
from sklearn.metrics import classification_report

# Load Stage 1 directly from HF
model_s1_eval = AutoModelForSequenceClassification.from_pretrained(
    "annasus10/xlmr-burmese-pragmatics-stage1"
)

# Build a trainer just for evaluation
from transformers import TrainingArguments, Trainer

eval_args = TrainingArguments(
    output_dir="./eval_tmp",
    per_device_eval_batch_size=32,
    report_to="none"
)

trainer_s1_eval = Trainer(
    model=model_s1_eval,
    args=eval_args,
    compute_metrics=compute_metrics
)

# Evaluate on test set
results_s1 = trainer_s1_eval.evaluate(tokenized_test)
print("\nStage 1 Test Results:")
for k, v in results_s1.items():
    print(f"  {k}: {v}")

# Classification report
preds_s1_out = trainer_s1_eval.predict(tokenized_test)
preds_s1 = np.argmax(preds_s1_out.predictions, axis=1)
true_s1 = preds_s1_out.label_ids

print("\nStage 1 Classification Report:")
print(classification_report(true_s1, preds_s1, target_names=le.classes_))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Stage 1 Test Results:
  eval_loss: 1.095820426940918
  eval_model_preparation_time: 0.0052
  eval_accuracy: 0.5727
  eval_macro_f1: 0.5451
  eval_weighted_f1: 0.5726
  eval_runtime: 2.2517
  eval_samples_per_second: 146.556
  eval_steps_per_second: 4.885

Stage 1 Classification Report:
              precision    recall  f1-score   support

       blunt       0.33      0.57      0.42        14
    informal       0.43      0.61      0.50        59
     neutral       0.87      0.44      0.58       174
      polite       0.50      0.91      0.64        56
professional       0.79      1.00      0.88        15
        rude       0.23      0.25      0.24        12

    accuracy                           0.57       330
   macro avg       0.53      0.63      0.55       330
weighted avg       0.68      0.57      0.57       330



In [28]:
import numpy as np
from sklearn.metrics import classification_report
import json, pickle
from google.colab import files

# ── Stage 2 evaluation ───────────────────────────────────────────────────────
results_s2 = trainer_s2.evaluate(tokenized_test_s2)
preds_s2_out = trainer_s2.predict(tokenized_test_s2)
preds_s2 = np.argmax(preds_s2_out.predictions, axis=1)
true_s2 = preds_s2_out.label_ids

print("Stage 2 Test Results:")
for k, v in results_s2.items():
    print(f"  {k}: {v}")
print("\nStage 2 Classification Report:")
print(classification_report(true_s2, preds_s2, target_names=le.classes_))

# ── Stage 3 evaluation ───────────────────────────────────────────────────────
results_s3 = trainer_s3.evaluate(tokenized_test_s3)
preds_s3_out = trainer_s3.predict(tokenized_test_s3)
preds_s3 = np.argmax(preds_s3_out.predictions, axis=1)
true_s3 = preds_s3_out.label_ids

print("\nStage 3 Test Results:")
for k, v in results_s3.items():
    print(f"  {k}: {v}")
print("\nStage 3 Classification Report:")
print(classification_report(true_s3, preds_s3, target_names=le.classes_))

# ── Save all small files ─────────────────────────────────────────────────────
# Label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

# Label mapping
with open("label_mapping.json", "w") as f:
    json.dump(label_mapping, f, indent=2)

# Classification reports
report_s2 = classification_report(true_s2, preds_s2, target_names=le.classes_, output_dict=True)
report_s3 = classification_report(true_s3, preds_s3, target_names=le.classes_, output_dict=True)
with open("report_stage2.json", "w") as f:
    json.dump(report_s2, f, indent=2)
with open("report_stage3.json", "w") as f:
    json.dump(report_s3, f, indent=2)

# Training curves
training_curves = {
    "stage1": {
        "epochs":     [1,2,3,4,5,6,7,8],
        "train_loss": [1.808,1.805,1.801,1.702,1.474,1.252,1.166,1.027],
        "val_loss":   [1.786,1.809,1.735,1.622,1.276,1.163,1.109,1.091],
        "macro_f1":   [0.045,0.049,0.131,0.263,0.474,0.495,0.545,0.534]
    },
    "stage2": {
        "epochs":     [1,2,3,4,5,6,7,8],
        "train_loss": [1.795,1.701,1.263,0.830,0.646,0.539,0.407,0.331],
        "val_loss":   [1.776,1.409,1.066,0.843,0.853,0.948,0.904,0.871],
        "macro_f1":   [0.051,0.375,0.491,0.579,0.651,0.620,0.636,0.650]
    },
    "stage3": {
        "epochs":     [1,2,3,4,5,6,7,8],
        "train_loss": [1.662,0.847,0.848,0.528,0.470,0.260,0.254,0.296],
        "val_loss":   [1.374,0.871,0.684,0.616,0.671,0.759,0.716,0.697],
        "macro_f1":   [0.476,0.583,0.669,0.728,0.754,0.789,0.800,0.821]
    }
}
with open("training_curves.json", "w") as f:
    json.dump(training_curves, f, indent=2)

# Dataset splits
train_data.to_csv("split_train.csv", index=False)
val_data.to_csv("split_val.csv", index=False)
test_data.to_csv("split_test.csv", index=False)

# Download all
for fname in ["label_encoder.pkl", "label_mapping.json",
              "report_stage2.json", "report_stage3.json",
              "training_curves.json",
              "split_train.csv", "split_val.csv", "split_test.csv"]:
    files.download(fname)
    print(f"Downloading {fname}...")

print("\nDone. All files queued.")

Stage 2 Test Results:
  eval_loss: 0.8274039626121521
  eval_accuracy: 0.7515
  eval_macro_f1: 0.728
  eval_weighted_f1: 0.7544
  eval_runtime: 2.3531
  eval_samples_per_second: 140.242
  eval_steps_per_second: 4.675
  epoch: 8.0

Stage 2 Classification Report:
              precision    recall  f1-score   support

       blunt       0.80      0.57      0.67        14
    informal       0.71      0.76      0.74        59
     neutral       0.86      0.73      0.79       174
      polite       0.59      0.82      0.69        56
professional       0.78      0.93      0.85        15
        rude       0.62      0.67      0.64        12

    accuracy                           0.75       330
   macro avg       0.73      0.75      0.73       330
weighted avg       0.77      0.75      0.75       330




Stage 3 Test Results:
  eval_loss: 0.9062870740890503
  eval_accuracy: 0.8697
  eval_macro_f1: 0.8248
  eval_weighted_f1: 0.8686
  eval_runtime: 2.5852
  eval_samples_per_second: 127.648
  eval_steps_per_second: 8.123
  epoch: 8.0

Stage 3 Classification Report:
              precision    recall  f1-score   support

       blunt       0.88      0.50      0.64        14
    informal       0.78      0.88      0.83        59
     neutral       0.94      0.87      0.90       174
      polite       0.78      0.95      0.85        56
professional       1.00      1.00      1.00        15
        rude       0.80      0.67      0.73        12

    accuracy                           0.87       330
   macro avg       0.86      0.81      0.82       330
weighted avg       0.88      0.87      0.87       330



Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done. All files queued.


In [29]:
import json
from google.colab import files
from sklearn.metrics import classification_report

# Stage 1 classification report
report_s1 = classification_report(
    true_s1, preds_s1,
    target_names=le.classes_,
    output_dict=True
)
with open("report_stage1.json", "w") as f:
    json.dump(report_s1, f, indent=2)

# Stage 1 numerical results
results_s1_dict = {
    "description": "Utterance only",
    "eval_loss": 1.0958,
    "eval_accuracy": 0.5727,
    "eval_macro_f1": 0.5451,
    "eval_weighted_f1": 0.5726
}
with open("results_stage1.json", "w") as f:
    json.dump(results_s1_dict, f, indent=2)

files.download("report_stage1.json")
files.download("results_stage1.json")
print("Stage 1 files downloaded.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Stage 1 files downloaded.


In [30]:
from huggingface_hub import HfApi
api = HfApi()

# ── Model card template ───────────────────────────────────────────────────────
card_s1 = """---
language: my
license: cc0-1.0
base_model: xlm-roberta-base
datasets:
  - freococo/burmese-contextual-pragmatics
tags:
  - text-classification
  - burmese
  - myanmar
  - pragmatics
  - xlm-roberta
  - fine-tuned
metrics:
  - accuracy
  - f1
---

# XLM-RoBERTa Burmese Pragmatics — Stage 1 (Utterance Only)

Fine-tuned version of `xlm-roberta-base` for **Burmese politeness classification**.
This is **Stage 1** of a 3-stage ablation study on pragmatic classification in Burmese.

## Task
Given a Burmese utterance, classify its politeness level into one of 6 classes:
`neutral`, `polite`, `informal`, `professional`, `blunt`, `rude`

## Dataset
[freococo/burmese-contextual-pragmatics](https://huggingface.co/datasets/freococo/burmese-contextual-pragmatics)
2,200 Burmese utterances covering 22 root meanings with pragmatic annotations.
Split: 70% train / 15% val / 15% test (seed=42)

## Stage Description
**Stage 1 — Utterance only.**
Input: raw Burmese utterance text with no additional context.
This is the weakest baseline in the ablation study.

## Results (Test Set)
| Metric | Score |
|---|---|
| Accuracy | 0.5727 |
| Macro F1 | 0.5451 |
| Weighted F1 | 0.5726 |
| Loss | 1.0958 |

## Per-class F1
| Class | Precision | Recall | F1 |
|---|---|---|---|
| blunt | 0.33 | 0.57 | 0.42 |
| informal | 0.43 | 0.61 | 0.50 |
| neutral | 0.87 | 0.44 | 0.58 |
| polite | 0.50 | 0.91 | 0.64 |
| professional | 0.79 | 1.00 | 0.88 |
| rude | 0.23 | 0.25 | 0.24 |

## Training
- Base model: `xlm-roberta-base`
- Epochs: 8
- Learning rate: 2e-5
- Batch size: 16
- Weighted cross-entropy loss (handles class imbalance)
- Best checkpoint selected by macro F1

## Ablation Study
| Stage | Input | Macro F1 |
|---|---|---|
| **Stage 1 (this model)** | Utterance only | **0.545** |
| Stage 2 | + context + instruction | see annasus10/xlmr-burmese-pragmatics-stage2 |
| Stage 3 | + register + power + tone | see annasus10/xlmr-burmese-pragmatics-stage3 |

## Citation
Group: AttentionIsAllUNeed — NLP Final Project
"""

card_s2 = """---
language: my
license: cc0-1.0
base_model: xlm-roberta-base
datasets:
  - freococo/burmese-contextual-pragmatics
tags:
  - text-classification
  - burmese
  - myanmar
  - pragmatics
  - xlm-roberta
  - fine-tuned
metrics:
  - accuracy
  - f1
---

# XLM-RoBERTa Burmese Pragmatics — Stage 2 (Utterance + Context)

Fine-tuned version of `xlm-roberta-base` for **Burmese politeness classification**.
This is **Stage 2** of a 3-stage ablation study on pragmatic classification in Burmese.

## Task
Given a Burmese utterance with social context, classify its politeness level into one of 6 classes:
`neutral`, `polite`, `informal`, `professional`, `blunt`, `rude`

## Dataset
[freococo/burmese-contextual-pragmatics](https://huggingface.co/datasets/freococo/burmese-contextual-pragmatics)
2,200 Burmese utterances covering 22 root meanings with pragmatic annotations.
Split: 70% train / 15% val / 15% test (seed=42)

## Stage Description
**Stage 2 — Utterance + context + instruction.**
Input format: `[utterance] </s> [context] </s> [instruction]`
Adds social situational context to the raw utterance.

## Results (Test Set)
| Metric | Score |
|---|---|
| Accuracy | 0.7485 |
| Macro F1 | 0.7056 |
| Weighted F1 | 0.7529 |
| Loss | 0.7066 |

## Per-class F1
| Class | Precision | Recall | F1 |
|---|---|---|---|
| blunt | 0.60 | 0.64 | 0.62 |
| informal | 0.69 | 0.76 | 0.73 |
| neutral | 0.89 | 0.73 | 0.80 |
| polite | 0.58 | 0.80 | 0.68 |
| professional | 0.83 | 1.00 | 0.91 |
| rude | 0.50 | 0.50 | 0.50 |

## Training
- Base model: `xlm-roberta-base`
- Epochs: 8
- Learning rate: 2e-5
- Batch size: 16
- Weighted cross-entropy loss (handles class imbalance)
- Best checkpoint selected by macro F1

## Ablation Study
| Stage | Input | Macro F1 |
|---|---|---|
| Stage 1 | Utterance only | see annasus10/xlmr-burmese-pragmatics-stage1 |
| **Stage 2 (this model)** | + context + instruction | **0.706** |
| Stage 3 | + register + power + tone | see annasus10/xlmr-burmese-pragmatics-stage3 |

## Citation
Group: AttentionIsAllUNeed — NLP Final Project
"""

card_s3 = """---
language: my
license: cc0-1.0
base_model: xlm-roberta-base
datasets:
  - freococo/burmese-contextual-pragmatics
tags:
  - text-classification
  - burmese
  - myanmar
  - pragmatics
  - xlm-roberta
  - fine-tuned
metrics:
  - accuracy
  - f1
---

# XLM-RoBERTa Burmese Pragmatics — Stage 3 (Full Pragmatic Input)

Fine-tuned version of `xlm-roberta-base` for **Burmese politeness classification**.
This is **Stage 3** of a 3-stage ablation study on pragmatic classification in Burmese.

## Task
Given a Burmese utterance with full pragmatic metadata, classify its politeness level into one of 6 classes:
`neutral`, `polite`, `informal`, `professional`, `blunt`, `rude`

## Dataset
[freococo/burmese-contextual-pragmatics](https://huggingface.co/datasets/freococo/burmese-contextual-pragmatics)
2,200 Burmese utterances covering 22 root meanings with pragmatic annotations.
Split: 70% train / 15% val / 15% test (seed=42)

## Stage Description
**Stage 3 — Full pragmatic input.**
Input format: `[register: X] [power: Y] [tone: Z] [utterance] </s> [context] </s> [instruction]`
Adds explicit pragmatic metadata (register, power relation, tone) as text prefix.

## Results (Test Set)
| Metric | Score |
|---|---|
| Accuracy | 0.8394 |
| Macro F1 | 0.8252 |
| Weighted F1 | 0.8421 |
| Loss | 0.5390 |

## Per-class F1
| Class | Precision | Recall | F1 |
|---|---|---|---|
| blunt | 0.83 | 0.71 | 0.77 |
| informal | 0.84 | 0.90 | 0.87 |
| neutral | 0.93 | 0.82 | 0.87 |
| polite | 0.66 | 0.86 | 0.74 |
| professional | 0.79 | 1.00 | 0.88 |
| rude | 0.90 | 0.75 | 0.82 |

## Training
- Base model: `xlm-roberta-base`
- Epochs: 8
- Learning rate: 2e-5
- Batch size: 8
- Weighted cross-entropy loss (handles class imbalance)
- Best checkpoint selected by macro F1

## Ablation Study
| Stage | Input | Macro F1 |
|---|---|---|
| Stage 1 | Utterance only | see annasus10/xlmr-burmese-pragmatics-stage1 |
| Stage 2 | + context + instruction | see annasus10/xlmr-burmese-pragmatics-stage2 |
| **Stage 3 (this model)** | + register + power + tone | **0.825** |

## Citation
Group: AttentionIsAllUNeed — NLP Final Project
"""

# ── Push all three cards ──────────────────────────────────────────────────────
api.upload_file(
    path_or_fileobj=card_s1.encode(),
    path_in_repo="README.md",
    repo_id="annasus10/xlmr-burmese-pragmatics-stage1",
    repo_type="model"
)
print("Stage 1 card updated.")

api.upload_file(
    path_or_fileobj=card_s2.encode(),
    path_in_repo="README.md",
    repo_id="annasus10/xlmr-burmese-pragmatics-stage2",
    repo_type="model"
)
print("Stage 2 card updated.")

api.upload_file(
    path_or_fileobj=card_s3.encode(),
    path_in_repo="README.md",
    repo_id="annasus10/xlmr-burmese-pragmatics-stage3",
    repo_type="model"
)
print("Stage 3 card updated.")

Stage 1 card updated.
Stage 2 card updated.
Stage 3 card updated.
